# 5. Parallel Performance with PCAIR

In the previous notebooks, we saw how AIR methods form strong F/C splittings and robust approximate ideal operators to handle asymmetric linear systems. In serial environments, we saw scalable, $\mathcal{O}(n)$ growth in work.

However, moving to large-scale parallel environments (both CPUs and GPUs) changes the story significantly. Achieving good weak scaling—where total time remains constant as problem size and compute resources increase proportionally—requires combating the communication and computation bottlenecks found in reduction multigrid hierarchies.

In [ ]:

import sys
import os
import re
import numpy as np
import matplotlib.pyplot as plt

import petsc4py
petsc4py.init(sys.argv)
from petsc4py import PETSc

import pflare

def build_2d_adv_diff(N, eps=1e-4):
    """2D advection-diffusion: -eps*Lap(u) + u_x = 1 with 5-point FD stencil."""
    h = 1.0 / (N + 1)
    n = N * N
    A = PETSc.Mat().create()
    A.setSizes([n, n])
    A.setFromOptions()
    A.setPreallocationNNZ(5)
    A.setUp()

    rstart, rend = A.getOwnershipRange()
    for row in range(rstart, rend):
        i, j = divmod(row, N)
        diag = 4.0 * eps / h**2 + 1.0 / h
        A.setValue(row, row, diag)
        if j > 0:     A.setValue(row, row - 1, -eps / h**2 - 1.0 / h)
        if j < N - 1: A.setValue(row, row + 1, -eps / h**2)
        if i > 0:     A.setValue(row, row - N, -eps / h**2)
        if i < N - 1: A.setValue(row, row + N, -eps / h**2)
    A.assemblyBegin(); A.assemblyEnd()

    b = A.createVecRight()
    b.set(1.0)
    return A, b

def _parse_air_stats(output):
    """Parse levels, operator complexity, and cycle complexity from PCAIR stats output."""
    levels = None
    op_cplx = None
    cyc_cplx = None
    m = re.search(r'Coarse grid\s+(\d+)', output)
    if m:
        levels = int(m.group(1))
    m = re.search(r'Operator complexity\s*:\s*([\d.Ee+\-]+)', output)
    if m:
        op_cplx = float(m.group(1))
    m = re.search(r'Cycle complexity\s*:\s*([\d.Ee+\-]+)', output)
    if m:
        cyc_cplx = float(m.group(1))
    return levels, op_cplx, cyc_cplx

def run_pcair(A, b, pc_options=None, max_it=500, rtol=1e-10):
    """Run PCAIR and return (iteration_count, num_levels, operator_complexity, cycle_complexity).

    Stats are always collected internally by capturing C-level stdout.
    If 'pc_air_print_stats_timings' is present in pc_options the full stats
    output is also printed to the notebook cell output.
    """
    pc_options = pc_options or {}
    print_stats = 'pc_air_print_stats_timings' in pc_options

    # Always enable stats internally so we can parse levels and complexities.
    opts_to_use = dict(pc_options)
    opts_to_use['pc_air_print_stats_timings'] = True

    opts = PETSc.Options()
    prefix = 'pcair5_'
    set_keys = []
    for key, value in opts_to_use.items():
        full_key = prefix + key
        if value is True or value is None:
            opts[full_key] = ''
        else:
            opts[full_key] = str(value)
        set_keys.append(full_key)

    result = []

    # Capture C-level stdout via fd-level redirection.
    sys.stdout.flush()
    old_fd = os.dup(1)
    r_fd, w_fd = os.pipe()
    os.dup2(w_fd, 1)
    os.close(w_fd)

    try:
        ksp = PETSc.KSP().create()
        ksp.setOptionsPrefix(prefix)
        ksp.setOperators(A)
        ksp.setType('gmres')
        ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
        ksp.setTolerances(rtol=rtol, max_it=max_it)
        ksp.getPC().setType('air')
        ksp.setFromOptions()
        x = A.createVecRight()
        x.set(0.0)
        ksp.solve(b, x)
        x.destroy()
        n_iters = ksp.getIterationNumber()
        if ksp.getConvergedReason() < 0:
            n_iters = max_it
        result.append(n_iters)
        ksp.destroy()
    finally:
        sys.stdout.flush()
        os.dup2(old_fd, 1)   # restore stdout
        os.close(old_fd)

    chunks = []
    while True:
        chunk = os.read(r_fd, 65536)
        if not chunk:
            break
        chunks.append(chunk)
    os.close(r_fd)
    captured = b''.join(chunks).decode('utf-8', errors='replace')

    for key in set_keys:
        try:
            del opts[key]
        except Exception:
            pass

    iters = result[0] if result else max_it
    levels, op_cplx, cyc_cplx = _parse_air_stats(captured)

    if print_stats:
        print(captured, end='')

    return iters, levels, op_cplx, cyc_cplx

print("Setup complete.")



## 1. The Challenge: Operator Densification

Standard geometric multigrids maintain the matrix sparsity pattern as they move down the hierarchy. Algebraic multigrids are different; often the number of non-zeros per row in the coarse grid matrix increases on the lower levels.

Reduction multigrids (as a type of algebraic multigrid) also suffer from this, but compared to multigrid methods for elliptic operators the problem is worse. Reduction multigrids in advection problems must coarsen slowly, giving many levels in the hierarchy. In elliptic multigrids, *aggressive coarsening* — tightening the strength-of-connection threshold so that coarser levels are reached in fewer steps — is a common way to limit how badly densification accumulates. For asymmetric problems this is not a viable option: aggressive coarsening significantly degrades convergence, so slow coarsening is unavoidable.

The coarse level operator is constructed via a Schur complement:
$$
S = A_{cc} - A_{cf}A_{ff}^{-1}A_{fc}.
$$
If we are using a first order fixed sparsity GMRES polynomial to approximate $A_{ff}^{-1}$, it will have the sparsity of $A_{ff}$. The term $A_{cf}A_{ff}^{-1}A_{fc}$ used to construct the coarse grid will therefore connect C points, to F points, to C points. This causes the stencil size on the coarse grid to grow. A matrix that had 5 non-zeros per row on the top level may quickly become very dense on level 3 or 4.


In [ ]:

# Demonstrate NNZ growth by manually constructing one coarse grid step.
# We use a red-black CF splitting (F points where (i+j) is odd) on a small 2D matrix.
# In PCAIR the CF splittings described in previous notebooks would be used
# This is just for simple illustration

N = 15
A, b = build_2d_adv_diff(N, eps=1e-4)
b.destroy()

n = N * N
nnz_A = int(A.getInfo()['nz_used'])

# --- Red-black CF splitting ---
# F points: (i+j) odd.  C points: (i+j) even.
f_indices = sorted([row for row in range(n) if ((row // N) + (row % N)) % 2 == 1])
c_indices = sorted([row for row in range(n) if ((row // N) + (row % N)) % 2 == 0])
nf, nc = len(f_indices), len(c_indices)

f_is = PETSc.IS().createGeneral(f_indices)
c_is = PETSc.IS().createGeneral(c_indices)

# --- Extract submatrices ---
A_ff = A.createSubMatrix(f_is, f_is)
A_fc = A.createSubMatrix(f_is, c_is)
A_cf = A.createSubMatrix(c_is, f_is)
A_cc = A.createSubMatrix(c_is, c_is)

nnz_A_cc = int(A_cc.getInfo()['nz_used'])

# --- Convert to numpy (feasible for this small size) ---
def petsc_to_np(mat):
    m, n = mat.getSize()
    return mat.getValues(range(m), range(n))

A_ff_np = petsc_to_np(A_ff)
A_fc_np = petsc_to_np(A_fc)
A_cf_np = petsc_to_np(A_cf)
A_cc_np = petsc_to_np(A_cc)

# --- Exact dense inverse of A_ff ---
# In PCAIR this would be an approximate inverse from PCPFLAREINV
A_ff_inv = np.linalg.inv(A_ff_np)

# --- Schur complement: S = A_cc - A_cf * A_ff^{-1} * A_fc ---
S = A_cc_np - A_cf_np @ A_ff_inv @ A_fc_np

# --- Count NNZs in S (entries above a relative tolerance) ---
tol = 1e-10 * np.abs(S).max()
nnz_S = int(np.sum(np.abs(S) > tol))

print(f"Matrix size:         {n} x {n}")
print(f"F points: {nf},  C points: {nc}")
print()
print(f"NNZ in A (top level):        {nnz_A:>6}   ({nnz_A/n:.2f} per row)")
print(f"NNZ in A_cc (C-C block):     {nnz_A_cc:>6}   ({nnz_A_cc/nc:.2f} per row)")
print(f"NNZ in S (exact Schur comp): {nnz_S:>6}   ({nnz_S/nc:.2f} per row)")
print()
print(f"S has {nnz_S/nnz_A:.1f}x the NNZs of the top-level matrix, "
      f"despite having only {nc/n:.0%} of the unknowns.")

# -- Visualise sparsity of A and S side by side --
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
A_np = petsc_to_np(A)
axes[0].spy(np.abs(A_np) > 0, markersize=2)
axes[0].set_title(f"$A$ sparsity  (NNZ={nnz_A})")
axes[1].spy(np.abs(S) > tol, markersize=2)
axes[1].set_title(f"Schur complement $S$ sparsity  (NNZ={nnz_S})")
plt.tight_layout()
plt.show()

for mat in [A_ff, A_fc, A_cf, A_cc, A]:
    mat.destroy()



The Schur complement above has around the same number of non-zeros as the top-level matrix, despite having only half the unknowns. Repeating this process level by level, the coarse matrices can quickly have *more* total non-zeros than the fine grid, even though they are smaller. 

This densification means that on each subsequent level, every rank must communicate with more neighbouring ranks when performing the SpGEMM operations that build the restrictor and coarse grid matrix. Once the off-process fraction of the matrix grows large enough, communication dominates completely.

---

## 2. CPUs and Processor Agglomeration

On CPUs a node typically has $O(100)$ cores, so a typical CPU-scale run might have $\sim 10^4$ DOFs per MPI rank. Reduction multigrids for advection problems coarsen slowly (at a rate of roughly $1/2$ in both 2D and 3D), giving many levels. After only a few coarsening steps the unknowns per rank drop below any reasonable threshold, while the NNZs per row in the coarse matrix keep growing. The SpGEMM products needed to form the restrictor and coarse grid become increasingly non-local: the local-to-non-local NNZ ratio drops well below 1, making those matrix--matrix products communication dominated.

**The Solution: Processor Agglomeration**

Rather than keeping all MPI ranks active on a nearly empty coarse grid, PCAIR can detect when the local-to-non-local NNZ ratio falls below a threshold and *agglomerate* unknowns: the data from several ranks is consolidated onto a smaller number of active ranks and the unknown partitioning modified to reduce communication with ParMETIS. This reduces the number of communicating processes, raises the DOFs-per-rank back to a healthy level, and brings the communication bottleneck back under control. 

This behaviour is controlled by several key parameters, including:

- `-pc_air_processor_agglom` — enable/disable agglomeration (default: `true`)
- `-pc_air_processor_agglom_ratio` — local/non-local NNZ ratio that triggers agglomeration (default: `2.0`)
- `-pc_air_processor_agglom_factor` — factor by which active ranks are reduced each time agglomeration triggers (default: `2`)



## 3. Controlling Densification: Drop Tolerances and Operator Complexity

Before looking at architecture-specific strategies, there is a simple option available on all platforms to keep coarse-grid matrix density in check.

**Operator complexity** is the sum of non-zeros across every matrix in the multigrid hierarchy, divided by the non-zeros in the top-level matrix. An operator complexity of 5 means you are storing the equivalent of 5 copies of the fine-grid matrix across all levels combined. Because reduction multigrids in advection problems coarsen slowly and the Schur complement densifies each coarse grid, the operator complexity can be considerably higher than in elliptic multigrid (where values of 1.5–2 are typical).

`-pc_air_a_drop` applies a relative drop tolerance to $A_\text{coarse}$ after it has been assembled on each level: any entry satisfying $|a_{ij}| \leq \epsilon \max_k |a_{ik}|$ is dropped and is either discarded or lumped onto the diagonal (`-pc_air_a_lump`). 

`-pc_air_r_drop` does the same for the restriction operator $R$. Both are applied on every level so their effect compounds down the hierarchy. A looser tolerance reduces operator complexity (and hence memory and inter-rank communication in the setup), at the cost of a less accurate coarse grid.

The default values (`-pc_air_a_drop 1e-4`, `-pc_air_r_drop 1e-2`) are already chosen to sparsify without noticeably degrading convergence. Below we vary `-pc_air_a_drop` and observe the operator complexity reported by `-pc_air_print_stats_timings`.


In [ ]:

# Vary pc_air_a_drop and observe its effect on operator complexity and iterations.
# Stats are captured internally; we extract just the operator complexity here.
# We use N=100 throughout.

A, b = build_2d_adv_diff(100)

print(f"{'a_drop':>10}  {'op_complexity':>15}  {'iterations':>11}")
for drop in [0.0, 1e-4, 1e-3]:
    iters, _, op_cplx, _ = run_pcair(A, b, pc_options={'pc_air_a_drop': drop})
    print(f"{drop:>10}  {op_cplx:>15.4f}  {iters:>11}")

A.destroy(); b.destroy()



## 4. GPUs: Truncation and Matrix-Free Polynomial Coarse Solvers

### The coarse grid problem

In any multigrid method, error modes not damped on the way down the hierarchy must be eliminated by the coarse grid solver. In classical AMG on elliptic problems the coarse grid is tiny and a small LU factorisation gathered onto one rank is adequate.

On GPUs this breaks down. GPUs have significant kernel launch overhead, so matrix operations on tiny systems cost as much in latency as in computation. Redundant distribution — every device holds a copy of the coarse grid, avoiding communication — still leaves each device working on a handful of rows, wasting nearly all available parallelism.

Reduction multigrids for advection compound this. Slow coarsening produces many levels, and densification makes the coarse matrices on the lower levels increasingly non-local. Setup SpGEMMs become communication-dominated; SpMVs at the coarsest levels can dominate runtime despite very little actual work.

We first inspect the full hierarchy that PCAIR builds by default.


In [ ]:

# --- Step 1: baseline ---
# Build a 100 x 100 advection-diffusion problem and run PCAIR with default settings.
# We print the setup statistics so we can see how many levels are built and how
# dense the coarse matrices become.
A, b = build_2d_adv_diff(100)

iters, levels, _, _ = run_pcair(A, b, pc_options={"pc_air_print_stats_timings": True})
print(f"\nBaseline: {levels} levels, converged in {iters} iterations")



### Truncating the hierarchy

The natural solution is to stop building the hierarchy before reaching the densest levels and replace everything below with a single coarse grid solve — called *truncation* (`-pc_air_max_levels`). This eliminates the expensive setup SpGEMMs on the densest levels, and removes those levels from the solve phase entirely.

Truncation does increase the work in the solve: the coarse grid is larger than it would otherwise be, so the coarse solve itself costs more per cycle — the cycle complexity rises. The key question is what kind of work that extra cost is. LU factorisation of the larger coarse grid would generate dense fill-in, consume large amounts of memory, and require triangle solves that communicate globally and scale poorly in parallel. Most standard coarse grid solvers are ruled out for exactly this reason.

Below we force truncation to different depths using `-pc_air_max_levels` with the default low-order Arnoldi polynomial coarse solver. The iteration count rises because the low-order polynomial is not strong enough to solve the larger truncated coarse grid.


In [ ]:

# --- Step 2: truncation with the default low-order coarse solver ---
# We force PCAIR to stop building coarse levels early using pc_air_max_levels.
# The default coarse solver is a low-order (order 6) Arnoldi polynomial, which is
# not powerful enough to solve the larger truncated coarse grid.  Iteration counts
# rise as we truncate more aggressively.
# We set max_levels explicitly so we already know the level count; just show iterations.
print("Hard truncation with default low-order coarse solver (arnoldi, order 6):")
print(f"{'max_levels':>12}  {'iterations':>11}  {'cyc_complexity':>15}")
for max_lev in [10, 7, 5]:
    iters, _, _, cyc_cplx = run_pcair(A, b, pc_options={
        "pc_air_max_levels": max_lev,
        "pc_air_coarsest_inverse_type": "arnoldi",
        "pc_air_coarsest_poly_order": 6,
    })
    print(f"{max_lev:>12}  {iters:>11}  {cyc_cplx:>15.4f}")



### GMRES polynomials as the coarse solver

The polynomial smoother from [notebook 3](03_cf_splitting.ipynb) approximated $A_{ff}^{-1}$ using a low-order Arnoldi polynomial on a diagonally dominant submatrix. For a coarse solver we need something more powerful: the full coarsest-level matrix may not be diagonally dominant, and approximating its inverse well typically requires orders of 50 or more.

The monomial basis $\{I,\, A,\, A^2,\, \ldots\}$ becomes numerically ill-conditioned at high order, making the coefficients unreliable.

**The Newton (harmonic Ritz) form** (`-pc_air_coarsest_inverse_type newton`) uses the harmonic Ritz values of the matrix as roots and remains stable at orders of 50, 100 or more. The polynomial is applied as a sequence of SpMVs against the coarsest-level matrix — no assembled inverse, no triangle solves, no global communication beyond the nearest-neighbour exchanges of a single SpMV. This is precisely the kind of work GPUs are designed for: repeated, regular, bandwidth-bound operations with only local data dependencies. The extra cost from truncating to a larger coarse grid is therefore very parallel-friendly, making truncation with a high-order Newton polynomial viable where LU would not be.


The same truncation levels that failed with the low-order Arnoldi coarse solver now converge robustly, at the cost of a higher cycle complexity.

In [ ]:

# --- Step 3: truncation with a high-order Newton polynomial coarse solver ---
# Switching to the Newton (harmonic Ritz) form permits stable high-order polynomials.
# With order-50 Newton polynomials as the coarse solver the iteration count is
# maintained even when we truncate to 5 levels -- the polynomial is strong enough
# to solve the larger, less-dense coarse grid.  No matrix is assembled: the coarse
# solver applies the polynomial via SpMVs of the coarse-level matrix.
# But it requires more work, the cycle complexity is much higher. In parallel however 
# this can still result in lower solve times
print("Hard truncation with high-order Newton polynomial coarse solver (newton, order 50):")
print(f"{'max_levels':>12}  {'iterations':>11}  {'cyc_complexity':>15}")
for max_lev in [10, 7, 5]:
    iters, _, _, cyc_cplx = run_pcair(A, b, pc_options={
        "pc_air_max_levels": max_lev,
        "pc_air_coarsest_inverse_type": "newton",
        "pc_air_coarsest_poly_order": 50,
        "pc_air_coarsest_matrix_free_polys": True,
    })
    print(f"{max_lev:>12}  {iters:>11}  {cyc_cplx:>15.4f}")



### Automatic truncation

For a given polynomial order, you do not know in advance which level is a safe truncation point. PCAIR can determine this automatically: starting from `-pc_air_auto_truncate_start_level` it tests the polynomial on each successive level, stopping the hierarchy as soon as the residual falls within `-pc_air_auto_truncate_tol`. A looser tolerance truncates earlier; tightening it pushes truncation deeper.


Truncating earlier raises the cycle complexity (more SpMVs in the coarse solve) but cuts setup cost and removes the densest levels from the solve. Whether the net effect is beneficial depends on the machine: on GPUs in parallel, where SpMVs are cheap and communication-per-SpMV is low, heavy truncation typically wins.

In [ ]:

# --- Step 4: automatic truncation ---
# Rather than setting max_levels by hand we let PCAIR determine the truncation
# point automatically.  Starting from level pc_air_auto_truncate_start_level it
# tests a tentative Newton polynomial coarse solver on each successive level.
# When the polynomial reduces a random residual to within pc_air_auto_truncate_tol
# the hierarchy is stopped there.
#
# A tighter tolerance pushes the truncation deeper (more levels, smaller coarse grid,
# easier for the polynomial); a looser tolerance truncates earlier (fewer levels,
# larger coarse grid, more work in the coarse solve but cheaper setup and fewer
# communication-heavy levels in the solve).

print(f"{'':20}  {'levels':>8}  {'iterations':>11}  {'cyc_complexity':>15}")

iters_full, levels_full, _, cyc_full = run_pcair(A, b, pc_options={
    "pc_air_coarsest_inverse_type": "newton",
    "pc_air_coarsest_poly_order": 50,
    "pc_air_coarsest_matrix_free_polys": True,
})
print(f"{'Full hierarchy':20}  {levels_full:>8}  {iters_full:>11}  {cyc_full:>15.4f}")

iters_b, levels_b, _, cyc_b = run_pcair(A, b, pc_options={
    "pc_air_coarsest_inverse_type": "newton",
    "pc_air_coarsest_poly_order": 50,
    "pc_air_coarsest_matrix_free_polys": True,
    "pc_air_auto_truncate_start_level": 2,
    "pc_air_auto_truncate_tol": 1e-6,
})
print(f"{'Auto tol=1e-6':20}  {levels_b:>8}  {iters_b:>11}  {cyc_b:>15.4f}")

iters_a, levels_a, _, cyc_a = run_pcair(A, b, pc_options={
    "pc_air_coarsest_inverse_type": "newton",
    "pc_air_coarsest_poly_order": 50,
    "pc_air_coarsest_matrix_free_polys": True,
    "pc_air_auto_truncate_start_level": 2,
    "pc_air_auto_truncate_tol": 1e-1,
})
print(f"{'Auto tol=1e-1':20}  {levels_a:>8}  {iters_a:>11}  {cyc_a:>15.4f}")

A.destroy(); b.destroy()


## Summary

PCAIR provides three main tools for controlling parallel performance, which can be used individually or in combination depending on the architecture:

- **Dropping**: removes small entries from the coarse operators at every level, reducing operator complexity (memory and communication) across the whole hierarchy.
- **Processor agglomeration**: consolidates work onto fewer active MPI ranks when too few DOFs remain per rank, keeping setup communication efficient. 
- **Truncation**: stops the hierarchy early and replaces the remaining levels with a high-quality coarse solver. Using a high-order GMRES polynomial in Newton form is an excellent fit.
